# NOWRI test drive

Explore the results of your corrected run: tables, maps, and a "what is near this place?" question box.

**Important: this notebook runs on its own Colab computer, so it cannot see the files of the main notebook.** It needs the GeoPackage file:

1. In the **main notebook**, run **cell 7B**. It saves `nowri_proximity_state.gpkg` to your computer (about 1-2 MB).
2. Run cell 1 below and choose that file when the picker appears.

(If the main notebook's session has ended, re-run its cells 1-3 and 7 first. The same test-drive cells are also built into the end of the main notebook, so you can use them there without any upload.)

Everything here is a **screening view**: distance to the nearest *mapped* river reach (about 500 m source resolution). It does not show exposure, contamination or impact.

### 1. Load the results

In [ ]:
import glob, os
import geopandas as gpd, pandas as pd, numpy as np, pyogrio
from google.colab import files

# A separate notebook runs on a separate computer, so it cannot see the main notebook's files.
paths = sorted(glob.glob("/content/nowri_out/state_riveratlas_*/nowri_proximity_state.gpkg")) + sorted(glob.glob("/content/*.gpkg"))
if not paths:
    print("Choose the file nowri_proximity_state.gpkg that you saved from the main notebook (cell 7B):")
    up = files.upload()
    paths = [os.path.join("/content", n) for n in up if n.lower().endswith(".gpkg")]
if not paths:
    raise SystemExit("No GeoPackage provided. See the instructions at the top of this notebook.")
GPKG = paths[-1]
print("Using:", GPKG)
layers = {str(n[0]): gpd.read_file(GPKG, layer=str(n[0])) for n in pyogrio.list_layers(GPKG)}
print({n: len(g) for n, g in layers.items()})

### 2. The headline tables
Rows are infrastructure types, columns are distance bands to the nearest mapped river. Only records inside Bayelsa, Rivers or Delta are counted.

In [ ]:
keep = ["OGIM_ID", "Asset_category", "SPATIAL_STATE", "Intersects_mapped_river", "Distance_to_river_m",
        "Distance_band", "Distance_band_to_pilot_network"]
frames = []
for n, g in layers.items():
    f = pd.DataFrame(g[[c for c in keep if c in g.columns]]).copy()
    f["layer"] = n
    frames.append(f)
allrec = pd.concat(frames, ignore_index=True)
assigned = allrec[allrec["SPATIAL_STATE"].notna()].copy()
assigned["Intersects_mapped_river"] = assigned["Intersects_mapped_river"].astype(bool)

order = ["Intersecting (0 km)", ">0\u20131 km", "1\u20135 km", ">5 km"]
by_type = pd.crosstab(assigned["Asset_category"], assigned["Distance_band"]).reindex(columns=order, fill_value=0)
by_type["Total"] = by_type.sum(axis=1)
display(by_type)
display(pd.crosstab(assigned["SPATIAL_STATE"], assigned["Distance_band"]).reindex(columns=order, fill_value=0))
print("assigned records:", len(assigned), "| intersecting:", int(assigned["Intersects_mapped_river"].sum()))

### 3. A quick map (static)
Colours show the distance band. There is no river layer on purpose: the river data are never copied into the outputs.

In [ ]:
import matplotlib.pyplot as plt
colors = {"Intersecting (0 km)": "#d62728", ">0\u20131 km": "#ff7f0e", "1\u20135 km": "#2ca02c", ">5 km": "#1f77b4"}
fig, ax = plt.subplots(figsize=(9, 9))
for n, g in layers.items():
    for band, col in colors.items():
        sub = g[g["Distance_band_to_pilot_network"] == band]
        if len(sub) == 0:
            continue
        kind = sub.geom_type.iloc[0]
        if "Point" in kind:
            sub.plot(ax=ax, color=col, markersize=6, alpha=0.8)
        elif "Line" in kind:
            sub.plot(ax=ax, color=col, linewidth=0.8, alpha=0.8)
        else:
            sub.plot(ax=ax, color=col, alpha=0.35, edgecolor=col, linewidth=0.3)
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=9, label=b)
           for b, c in colors.items()]
ax.legend(handles=handles, title="Distance to nearest mapped river", loc="lower left")
ax.set_title("Infrastructure by distance band (corrected extent)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.show()

### 4. An interactive map (zoom, pan, click for details)
Use the layer button (top right) to switch layers on and off, and to switch to a satellite background. Click any feature to see its details.

In [ ]:
!pip install -q folium mapclassify
import folium
band_order = ["Intersecting (0 km)", ">0\u20131 km", "1\u20135 km", ">5 km"]
band_colors = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]
show = ["oil_gas_fields", "pipelines", "flaring", "stations", "offshore", "terminals",
        "refineries", "lng", "gas_processing", "wells"]
cols = ["OGIM_ID", "Asset_category", "Asset_subtype", "SPATIAL_STATE",
        "Distance_band_to_pilot_network", "Distance_to_river_m", "geometry"]
m = None
for n in show:
    if n not in layers:
        continue
    g = layers[n]
    g = g[[c for c in cols if c in g.columns]]
    m = g.explore(m=m, name=n, column="Distance_band_to_pilot_network", categorical=True,
                  categories=band_order, cmap=band_colors, legend=(m is None),
                  tiles="CartoDB positron", marker_kwds={"radius": 4}, style_kwds={"weight": 1})
# Satellite basemap (switch it on in the layer control, top right). Esri imagery; keep the attribution.
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri, Maxar, Earthstar Geographics", name="Satellite (Esri)", overlay=False).add_to(m)
folium.LayerControl().add_to(m)
m

### 5. Ask: what is near this place?
Type a latitude/longitude and a radius in km. Try the examples, then your own place of interest.
(The coordinates below are approximate town centres.)

In [ ]:
AEQD = "+proj=aeqd +lat_0=5.25 +lon_0=5.75 +datum=WGS84 +units=m +no_defs"

def near(lat, lon, km=10):
    p = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs=4326).to_crs(AEQD).iloc[0]
    rows = []
    for name, g in layers.items():
        d_km = g.to_crs(AEQD).distance(p) / 1000
        hit = d_km <= km
        if hit.any():
            sub = pd.DataFrame(g.loc[hit].drop(columns="geometry"))
            sub["layer"] = name
            sub["km_from_point"] = d_km[hit].round(2)
            rows.append(sub)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True).sort_values("km_from_point")
    cols = ["layer", "Asset_category", "Asset_subtype", "SPATIAL_STATE", "km_from_point",
            "Distance_band", "Distance_to_river_m", "OGIM_ID"]
    return out[[c for c in cols if c in out.columns]]

places = {"Port Harcourt": (4.82, 7.03), "Bonny": (4.43, 7.17), "Yenagoa": (4.92, 6.26), "Warri": (5.52, 5.76)}
for name, (la, lo) in places.items():
    r = near(la, lo, 15)
    print(f"{name}: {len(r)} records within 15 km;",
          "by type:", r["layer"].value_counts().to_dict() if len(r) else {})
display(near(4.82, 7.03, 15).head(15))      # change the numbers to try your own place

### 6. How fragile is the "intersecting" count?
A record only counts as intersecting if its geometry touches a mapped river line exactly. The river lines are coarse (about 500 m),
so this curve shows how the share changes if you allow a small positional tolerance.

In [ ]:
tols = [0, 1, 10, 50, 100, 250, 500, 1000]
share = [100 * float((assigned["Intersects_mapped_river"] | (assigned["Distance_to_river_m"] <= t)).mean())
         for t in tols]
plt.figure(figsize=(7, 4))
plt.plot(range(len(tols)), share, marker="o")
plt.xticks(range(len(tols)), [f"{t} m" for t in tols])
plt.ylabel("% of assigned records"); plt.xlabel("Positional tolerance")
plt.title("Intersecting or within a tolerance of a mapped river")
plt.grid(alpha=0.3); plt.show()
print({f"{t} m": round(s, 1) for t, s in zip(tols, share)})

### 7. Try your own question
A few starting points: edit and run.

In [ ]:
# Gas flaring detections within 250 m of a mapped river, by state
q = assigned[(assigned["layer"] == "flaring") & (assigned["Distance_to_river_m"] <= 250)]
display(q["SPATIAL_STATE"].value_counts())

# Pipeline sections that intersect a mapped river: count and total length per state (sections, not crossings)
pl = layers["pipelines"]
display(pl[pl["Intersects_mapped_river"].astype(bool)].groupby("SPATIAL_STATE")["CLIPPED_LENGTH_KM"]
          .agg(["count", "sum"]).round(1))

# Records where OGIM's own state attribute disagrees with the spatially derived state
for n in ["stations", "offshore", "terminals", "flaring"]:
    g = layers.get(n)
    if g is not None and "State_attribute_conflict" in g.columns:
        print(n, "state attribute conflicts:", int(g["State_attribute_conflict"].sum()))

### 8. Take it into QGIS (optional, free desktop GIS)
This downloads the GeoPackage so you can open it in QGIS. **It contains third-party-derived data: keep it out of GitHub.**
In QGIS: *Layer, Add Layer, Add Vector Layer*, choose the file, tick all layers, then style by the field `Distance_band_to_pilot_network`
(*Layer Properties, Symbology, Categorized*). Add a base map from *Browser panel, XYZ Tiles, OpenStreetMap*.

In [ ]:
from google.colab import files
files.download(GPKG)

### Reading the results responsibly
- **Proximity is not exposure.** A flare 200 m from a mapped creek says nothing about water quality.
- **The river data are coarse and incomplete.** Many small channels, creeks and tidal channels in the delta are not mapped.
- **OGIM is patchy.** Status and operator are "N/A" for every record; no onshore wells appear.
- **Distances beyond the three state polygons are to a clipped river network** (see the `Location_assignment` field).